# Validasi Data SPPG Jawa Timur

## Import

In [43]:
import csv
import math
import re
import time
import sys
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
from urllib.parse import quote_plus

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

## Konfigurasi parameter

In [ ]:
# PATH
INPUT_PATH   = Path("../data/sppg_jawa_timur (1).csv") 
OUTPUT_PATH  = Path("outputs") / f"sppg_validated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
PROFILE_DIR  = "selenium_chrome_profile" 

# SELENIUM
HEADLESS     = False  
DELAY        = 4.0     
TIMEOUT      = 30     

# VALIDASI
DISTANCE_THRESHOLD_M = 500.0   
NAME_THRESHOLD       = 0.55    
MAX_CANDIDATES       = 3     

# RESUME / BATCHING
START_ROW    = 1    
LIMIT        = 0    
RESUME       = False  

print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_PATH}")
print(f"Headless: {HEADLESS} | Delay: {DELAY}s | Timeout: {TIMEOUT}s")
print(f"Distance threshold: {DISTANCE_THRESHOLD_M}m | Name threshold: {NAME_THRESHOLD}")

Input  : ..\data\sppg_jawa_timur (1).csv
Output : outputs\sppg_validated_20260508_125058.csv
Headless: False | Delay: 4.0s | Timeout: 30s
Distance threshold: 500.0m | Name threshold: 0.55


## Konstanta dan utility function

In [45]:
RESULT_COLUMNS = [
    "Status", "GMaps_Nama", "GMaps_Alamat",
    "GMaps_Longitude", "GMaps_Latitude",
    "GMaps_Rating", "GMaps_Reviews",
    "GMaps_Foto", "GMaps_Foto_URL",
    "GMaps_Distance_Meter", "GMaps_Name_Score",
    "GMaps_Source", "GMaps_Query",
    "GMaps_Candidate_Count", "GMaps_Filtered_Candidate_Count",
    "GMaps_Is_SPPG_Like", "GMaps_Error",
]

COORD_RE = re.compile(r"^-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?$")

SPPG_LIKE_PHRASES = [
    "sppg",
    "satuan pelayanan pemenuhan gizi",
    "pelayanan pemenuhan gizi",
    "satuan pelayanan gizi",
    "pemenuhan gizi",
    "dapur mbg",
    "mbg",
    "dapur bgn",       
    "bgn",             
    "makan bergizi",   
]

In [46]:
# Teks & Normalisasi

def clean_text(value):
    """Hapus whitespace berlebih."""
    return re.sub(r"\s+", " ", (value or "").strip())

def normalize_name(value):
    """Normalisasi nama untuk perbandingan: lowercase, buang stopword domain."""
    value = clean_text(value).lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    stopwords = (
        r"\b(sppg|satuan|pelayanan|pemenuhan|gizi|dapur|mbg|bgn|"
        r"program|makan|bergizi|gratis|kab|kabupaten|kec|kecamatan|"
        r"desa|kelurahan|rt|rw)\b"  
    )
    value = re.sub(stopwords, " ", value)
    return clean_text(value)

def normalize_keyword_text(value):
    value = clean_text(value).lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    return clean_text(value)

def is_sppg_like_text(value):
    """Cek apakah teks mengandung penanda SPPG/MBG."""
    text = normalize_keyword_text(value)
    if not text:
        return False
    if any(phrase in text for phrase in SPPG_LIKE_PHRASES):
        return True
    tokens = set(text.split())
    return (
        {"satuan", "pelayanan", "gizi"}.issubset(tokens)
        or {"pemenuhan", "gizi"}.issubset(tokens)
    )

def place_is_sppg_like(place):
    combined = " ".join([
        place.get("candidate_name", ""),
        place.get("name", ""),
        place.get("address", ""),
    ])
    return is_sppg_like_text(combined)

# --- Quick test ---
assert is_sppg_like_text("Dapur MBG Sukamaju") == True
assert is_sppg_like_text("Warung Makan Pak Budi") == False
print("   Tes is_sppg_like_text: LULUS")

   Tes is_sppg_like_text: LULUS


In [47]:
# Skor Kemiripan Nama

def levenshtein_distance(left, right):
    if left == right:
        return 0
    if not left:
        return len(right)
    if not right:
        return len(left)
    previous = list(range(len(right) + 1))
    for i, lc in enumerate(left, 1):
        current = [i]
        for j, rc in enumerate(right, 1):
            current.append(min(current[j-1]+1, previous[j]+1, previous[j-1]+(lc!=rc)))
        previous = current
    return previous[-1]

def name_score(expected, actual):
    """
    Gabungan tiga metode scoring:
    - Token overlap
    - SequenceMatcher ratio
    - Levenshtein similarity
    Mengembalikan nilai tertinggi dari ketiganya (0-1).
    """
    expected_norm = normalize_name(expected)
    actual_norm   = normalize_name(actual)
    if not expected_norm or not actual_norm:
        return 0.0
    if expected_norm in actual_norm or actual_norm in expected_norm:
        return 1.0

    exp_tokens = set(expected_norm.split())
    act_tokens = set(actual_norm.split())
    token_score       = len(exp_tokens & act_tokens) / max(len(exp_tokens), 1)
    ratio_score       = SequenceMatcher(None, expected_norm, actual_norm).ratio()
    max_len           = max(len(expected_norm), len(actual_norm), 1)
    levenshtein_score = 1 - (levenshtein_distance(expected_norm, actual_norm) / max_len)
    return max(token_score, ratio_score, levenshtein_score)

# --- Quick test ---
score_a = name_score("SPPG Sukamaju", "Dapur MBG Sukamaju")
score_b = name_score("SPPG Sukamaju", "Bengkel Motor Jaya")
print(f"   name_score('SPPG Sukamaju', 'Dapur MBG Sukamaju') = {score_a:.2f}  (harap > 0.3)")
print(f"   name_score('SPPG Sukamaju', 'Bengkel Motor Jaya')  = {score_b:.2f}  (harap < 0.3)")

   name_score('SPPG Sukamaju', 'Dapur MBG Sukamaju') = 1.00  (harap > 0.3)
   name_score('SPPG Sukamaju', 'Bengkel Motor Jaya')  = 0.23  (harap < 0.3)


In [48]:
# Utilitas Geografis & Parsing

def haversine_m(lat1, lon1, lat2, lon2):
    """Hitung jarak dua titik koordinat dalam meter."""
    R  = 6_371_000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1-a))

def parse_float(value):
    """Parse string ke float, return None jika gagal."""
    try:
        return float(str(value).strip().replace(",", ".")) 
    except (TypeError, ValueError):
        return None

## Scraping

In [49]:
def build_driver(headless=HEADLESS, profile_dir=PROFILE_DIR, timeout=TIMEOUT):
    """Buat instance Chrome WebDriver."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--lang=id-ID")
    options.add_argument("--window-size=1366,900")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    if profile_dir:
        Path(profile_dir).mkdir(parents=True, exist_ok=True)
        options.add_argument(f"--user-data-dir={Path(profile_dir).resolve()}")
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(timeout)
    return driver

def wait_for_maps(driver, timeout=TIMEOUT):
    wait = WebDriverWait(driver, timeout)
    wait.until(lambda d: "google." in d.current_url.lower() or "maps" in d.title.lower())
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

def click_consent_if_present(driver):
    """Klik tombol consent Google jika muncul."""
    labels = ["Accept all", "I agree", "Saya setuju", "Terima semua", "Setuju"]
    for label in labels:
        xpath = f"//button//*[contains(normalize-space(.), '{label}')]/ancestor::button"
        buttons = driver.find_elements(By.XPATH, xpath)
        if buttons:
            try:
                buttons[0].click()
                time.sleep(2)
                return
            except WebDriverException:
                pass

def load_maps_url(driver, url, timeout=TIMEOUT, delay=DELAY):
    """Load URL Google Maps dan tunggu halaman siap."""
    driver.get(url)
    wait_for_maps(driver, timeout)
    click_consent_if_present(driver)
    time.sleep(delay)

In [50]:
# Ekstraksi Data dari Halaman GMaps

def first_text(driver, selectors):
    for by, selector in selectors:
        for el in driver.find_elements(by, selector):
            text = clean_text(el.text or el.get_attribute("aria-label"))
            if text:
                return text
    return ""

def extract_title(driver):
    title = first_text(driver, [
        (By.CSS_SELECTOR, "h1.DUwDvf"),
        (By.CSS_SELECTOR, "h1"),
        (By.CSS_SELECTOR, "[role='main'] h1"),
    ])
    title = re.sub(r"^Hasil untuk\s+", "", title, flags=re.IGNORECASE)
    if COORD_RE.match(title) or title.lower() in {"dropped pin", "pin dipasang"}:
        return ""
    return title

def extract_address(driver):
    selectors = [
        (By.CSS_SELECTOR, "button[data-item-id='address']"),
        (By.CSS_SELECTOR, "button[aria-label^='Alamat:']"),
        (By.CSS_SELECTOR, "button[aria-label^='Address:']"),
        (By.XPATH, "//*[contains(@aria-label, 'Alamat:') or contains(@aria-label, 'Address:')]"),
    ]
    text = first_text(driver, selectors)
    return clean_text(re.sub(r"^(Alamat|Address):\s*", "", text, flags=re.IGNORECASE))

def extract_rating(driver):
    selectors = [
        (By.CSS_SELECTOR, "div.F7nice span[aria-hidden='true']"),
        (By.CSS_SELECTOR, "span.MW4etd"),
        (By.XPATH, "//*[contains(@aria-label, 'bintang') or contains(@aria-label, 'stars')]"),
    ]
    for by, selector in selectors:
        for el in driver.find_elements(by, selector):
            text = clean_text(el.text or el.get_attribute("aria-label"))
            m = re.search(r"(\d+(?:[,.]\d+)?)", text)
            if m:
                return m.group(1).replace(",", ".")
    return ""

def parse_review_count(text):
    text = clean_text(text).lower()
    for pattern in [
        r"([\d.,]+)\s*(?:ulasan|review|reviews)",
        r"\(([\d.,]+)\)",
    ]:
        m = re.search(pattern, text)
        if m:
            try:
                return int(m.group(1).replace(".", "").replace(",", ""))
            except ValueError:
                pass
    return None

def extract_review_count(driver):
    selectors = [
        (By.CSS_SELECTOR, "button.HHrUdb"),
        (By.CSS_SELECTOR, "span.UY7F9"),
        (By.CSS_SELECTOR, "div.F7nice"),
        (By.XPATH, "//*[contains(text(), 'ulasan') or contains(text(), 'review') or contains(text(), 'reviews')]"),
        (By.XPATH, "//*[contains(@aria-label, 'ulasan') or contains(@aria-label, 'review') or contains(@aria-label, 'reviews')]"),
    ]
    for by, selector in selectors:
        for el in driver.find_elements(by, selector):
            text = clean_text(el.text or el.get_attribute("aria-label"))
            parsed = parse_review_count(text)
            if parsed is not None:
                return parsed
    return 0

def extract_photo(driver):
    selectors = [
        "button[jsaction*='pane.heroHeaderImage'] img",
        "button[aria-label*='Foto'] img",
        "button[aria-label*='Photo'] img",
        "[role='main'] img[src*='googleusercontent']",
    ]
    for selector in selectors:
        for el in driver.find_elements(By.CSS_SELECTOR, selector):
            src = el.get_attribute("src") or ""
            if src and not src.startswith("data:"):
                return "ADA", src
    return "TIDAK ADA", ""

def extract_coords_from_url(url):
    """Ekstrak koordinat dari URL Google Maps."""
    # Format !3d<lat>!4d<lon> lebih akurat dari @lat,lon
    m = re.search(r"!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)", url)
    if m:
        return float(m.group(1)), float(m.group(2))
    m = re.search(r"@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)", url)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

def scrape_current_place(driver):
    """Ekstrak semua data dari halaman detail tempat yang sedang terbuka."""
    return {
        "name"         : extract_title(driver),
        "address"      : extract_address(driver),
        "rating"       : extract_rating(driver),
        "review_count" : extract_review_count(driver),
        "photo_status" : extract_photo(driver)[0],
        "photo_url"    : extract_photo(driver)[1],
        "lat"          : extract_coords_from_url(driver.current_url)[0],
        "lon"          : extract_coords_from_url(driver.current_url)[1],
    }

def has_place_data(place):
    return bool(
        place["name"] or place["address"] or
        place["rating"] or place["review_count"] or place["photo_url"]
    )

In [51]:
# Pencarian & Pemilihan Kandidat

def extract_search_results(driver, max_results=MAX_CANDIDATES):
    """
    Ambil daftar hasil pencarian dari halaman search GMaps.
    """
    results = []
    seen    = set()
    links   = driver.find_elements(
        By.CSS_SELECTOR,
        "a[href*='/maps/place/'], a[href*='www.google.com/maps/place/']"
    )
    for link in links:
        href = link.get_attribute("href") or ""
        if not href or href in seen:
            continue
        label = clean_text(link.get_attribute("aria-label") or link.text)
        if not label:
            continue
        label = label.split("\n")[0]
        if COORD_RE.match(label):
            continue
        seen.add(href)
        results.append({"name": label, "href": href})
        if len(results) >= max_results:
            break
    return results

def build_search_query(row):
    """
    PERBAIKAN: Tambah konteks kota/kabupaten ke query agar lebih presisi.
    Gunakan kolom 'Kabupaten'/'Kota' jika tersedia di CSV.
    """
    name = clean_text(row.get("Nama_SPPG", ""))
    kota = clean_text(row.get("Kabupaten", "") or row.get("Kota", "") or row.get("Kecamatan", ""))
    if kota:
        return f"{name} {kota}"
    return name

def scrape_search_candidates(driver, row):
    """
    Jalankan pencarian GMaps lalu scrape setiap kandidat.
    Return: list of place dicts.
    """
    query = build_search_query(row)
    url   = f"https://www.google.com/maps/search/?api=1&query={quote_plus(query)}"
    load_maps_url(driver, url)

    candidates = extract_search_results(driver, MAX_CANDIDATES)
    places = []

    if candidates:
        for idx, cand in enumerate(candidates, 1):
            load_maps_url(driver, cand["href"])
            place = scrape_current_place(driver)
            if not place["name"]:
                place["name"] = cand["name"]
            place["candidate_name"]  = cand["name"]
            place["is_sppg_like"]    = place_is_sppg_like(place)
            place["source"]          = f"search_result_{idx}"
            place["query"]           = query
            place["candidate_count"] = len(candidates)
            places.append(place)
    else:
        # GMaps langsung arahkan ke satu tempat
        place = scrape_current_place(driver)
        place["candidate_name"]  = ""
        place["is_sppg_like"]    = place_is_sppg_like(place)
        place["source"]          = "search_direct"
        place["query"]           = query
        place["candidate_count"] = 1 if has_place_data(place) else 0
        places.append(place)

    return places

def choose_best_candidate(row, places):
    """
    Pilih kandidat terbaik berdasarkan:
    1. Filter: hanya yang SPPG-like
    2. Rank: berdasarkan name_score
    """
    expected_name    = row.get("Nama_SPPG", "")
    filtered_places  = [p for p in places if p.get("is_sppg_like")]
    candidate_pool   = filtered_places if filtered_places else places

    # Tandai filtered_candidate_count di semua place
    for p in places:
        p["filtered_candidate_count"] = len(filtered_places)

    scored = sorted(
        [(name_score(expected_name, p["name"]), p) for p in candidate_pool],
        key=lambda x: x[0], reverse=True
    )

    empty = {
        "name": "", "address": "", "rating": "", "review_count": 0,
        "photo_status": "TIDAK ADA", "photo_url": "", "lat": None, "lon": None,
        "candidate_name": "", "is_sppg_like": False,
        "filtered_candidate_count": 0,
        "query": build_search_query(row), "candidate_count": 0,
    }

    if not scored:
        return {**empty, "source": "search_no_result"}

    if len(places) > 1 and not filtered_places:
        return {**empty, "source": "search_no_sppg_like_result", "candidate_count": len(places)}

    best_score, best_place = scored[0]
    if best_score < NAME_THRESHOLD:
        best_place["source"] = f"{best_place['source']}_low_similarity"
    return best_place

## Fungsi evaluasi

In [52]:
def evaluate(row, place):
    """
    Evaluasi apakah place yang ditemukan sesuai dengan data CSV.
    Return dict hasil validasi.
    """
    source_lat    = parse_float(row.get("Latitude"))
    source_lon    = parse_float(row.get("Longitude"))
    expected_name = row.get("Nama_SPPG", "")

    score    = name_score(expected_name, place["name"])
    distance = None
    if all(v is not None for v in [source_lat, source_lon, place["lat"], place["lon"]]):
        distance = haversine_m(source_lat, source_lon, place["lat"], place["lon"])

    errors = []
    if not has_place_data(place):
        errors.append("Tidak ada data tempat terdeteksi di Google Maps")
    if not place["name"]:
        errors.append("Nama tempat tidak terdeteksi")
    if not place.get("is_sppg_like"):
        errors.append("Nama/alamat kandidat tidak mengandung SPPG atau Satuan Pelayanan Pemenuhan Gizi")
    if score < NAME_THRESHOLD:
        errors.append(f"Nama tidak cocok dengan CSV (score {score:.2f})")
    if place["photo_status"] == "TIDAK ADA" and int(place.get("review_count") or 0) == 0:
        errors.append("Tidak ada foto dan 0 review")
    if distance is None:
        errors.append("Koordinat tempat dari Google Maps tidak terdeteksi")
    elif distance > DISTANCE_THRESHOLD_M:
        errors.append(f"Jarak koordinat {distance:.1f} m melebihi ambang {DISTANCE_THRESHOLD_M:.1f} m")

    status = "VALID" if not errors else "TIDAK VALID"
    return {
        "Status"                        : status,
        "GMaps_Nama"                    : place["name"],
        "GMaps_Alamat"                  : place["address"],
        "GMaps_Longitude"               : "" if place["lon"] is None else f"{place['lon']:.8f}",
        "GMaps_Latitude"                : "" if place["lat"] is None else f"{place['lat']:.8f}",
        "GMaps_Rating"                  : place["rating"],
        "GMaps_Reviews"                 : str(place.get("review_count", 0)),
        "GMaps_Foto"                    : place["photo_status"],
        "GMaps_Foto_URL"                : place["photo_url"],
        "GMaps_Distance_Meter"          : "" if distance is None else f"{distance:.1f}",
        "GMaps_Name_Score"              : f"{score:.2f}",
        "GMaps_Source"                  : place["source"],
        "GMaps_Query"                   : place.get("query", ""),
        "GMaps_Candidate_Count"         : str(place.get("candidate_count", "")),
        "GMaps_Filtered_Candidate_Count": str(place.get("filtered_candidate_count", "")),
        "GMaps_Is_SPPG_Like"            : "YA" if place.get("is_sppg_like") else "TIDAK",
        "GMaps_Error"                   : "; ".join(errors),
    }

In [53]:
# I/O CSV

def read_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows   = list(reader)
    return reader.fieldnames or [], rows

def processed_count(output_path):
    if not Path(output_path).exists():
        return 0
    with open(output_path, "r", encoding="utf-8-sig", newline="") as f:
        return sum(1 for _ in csv.DictReader(f))

def output_header_fields(output_path):
    if not Path(output_path).exists():
        return []
    with open(output_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or []

def prepare_writer(output_path, fieldnames, append=False):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    handle = open(output_path, "a" if append else "w", encoding="utf-8-sig", newline="")
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    if not append:
        writer.writeheader()
    return handle, writer

## Pipline utama

In [54]:
# Validasi input CSV dulu sebelum membuka browser

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"File CSV tidak ditemukan: {INPUT_PATH}")

source_fields, rows = read_rows(INPUT_PATH)
required = {"Nama_SPPG", "Longitude", "Latitude"}
missing  = sorted(required - set(source_fields))
if missing:
    raise ValueError(f"Kolom wajib tidak ada di CSV: {', '.join(missing)}")

fieldnames = source_fields + [c for c in RESULT_COLUMNS if c not in source_fields]

# Tentukan baris yang akan diproses
start_index = max(START_ROW - 1, 0)
if RESUME:
    start_index = max(start_index, processed_count(OUTPUT_PATH))
selected_rows = rows[start_index:]
if LIMIT:
    selected_rows = selected_rows[:LIMIT]

append = RESUME and OUTPUT_PATH.exists() and processed_count(OUTPUT_PATH) > 0

print(f"Total baris CSV     : {len(rows)}")
print(f"Mulai dari baris    : {start_index + 1}")
print(f"Baris yang diproses : {len(selected_rows)}")
print(f"Mode output         : {'APPEND' if append else 'BARU'}")

Total baris CSV     : 3996
Mulai dari baris    : 1
Baris yang diproses : 50
Mode output         : BARU


In [55]:
handle, writer = prepare_writer(OUTPUT_PATH, fieldnames, append=append)
driver = build_driver()

try:
    total = len(selected_rows)
    for offset, row in enumerate(selected_rows, start=1):
        row_number = start_index + offset
        name       = row.get("Nama_SPPG", "")
        lat        = parse_float(row.get("Latitude"))
        lon        = parse_float(row.get("Longitude"))
        result     = {col: "" for col in RESULT_COLUMNS}

        print(f"[{offset}/{total}] Baris {row_number}: {name}")

        try:
            if lat is None or lon is None:
                result["Status"]    = "TIDAK VALID"
                result["GMaps_Error"] = "Longitude/Latitude CSV tidak valid"
            else:
                places = scrape_search_candidates(driver, row)
                place  = choose_best_candidate(row, places)
                result.update(evaluate(row, place))

        except TimeoutException:
            result["Status"]    = "TIDAK VALID"
            result["GMaps_Error"] = "Timeout saat membuka Google Maps"
        except WebDriverException as exc:
            result["Status"]    = "TIDAK VALID"
            result["GMaps_Error"] = f"Selenium error: {clean_text(str(exc))[:300]}"
        except Exception as exc:
            result["Status"]    = "TIDAK VALID"
            result["GMaps_Error"] = f"Error tidak terduga: {clean_text(str(exc))[:300]}"

        output_row = {**row, **result}
        writer.writerow(output_row)
        handle.flush()
        print(f"    → {result['Status']} | {result.get('GMaps_Nama', '')} | {result.get('GMaps_Error', '')}")

finally:
    handle.close()
    driver.quit()
    print(f"\nSelesai. Output: {OUTPUT_PATH.resolve()}")

[1/50] Baris 1: SPPG Madiun Geger Sumberejo
    → TIDAK VALID | SPPG MADIUN KEBONSARI REJOSARI | Jarak koordinat 6533.8 m melebihi ambang 500.0 m
[2/50] Baris 2: SPPG Malang Sumberpucung Karangkates
    → TIDAK VALID |  | Nama tempat tidak terdeteksi; Nama/alamat kandidat tidak mengandung SPPG atau Satuan Pelayanan Pemenuhan Gizi; Nama tidak cocok dengan CSV (score 0.00)
[3/50] Baris 3: SPPG Malang Ngantang Sumberagung 2
    → TIDAK VALID | SPPG Sumberagung 01 Ngantang | Nama tidak cocok dengan CSV (score 0.50); Jarak koordinat 913.2 m melebihi ambang 500.0 m
[4/50] Baris 4: SPPG Sidoarjo Waru Pepelegi
    → VALID | SPPG PEPELEGI | 
[5/50] Baris 5: SPPG Gresik Driyorejo Driyorejo 2
    → TIDAK VALID | SPPG Tanjungan | Nama tidak cocok dengan CSV (score 0.05); Jarak koordinat 2048.1 m melebihi ambang 500.0 m
[6/50] Baris 6: SPPG Kota Pasuruan Panggungrejo Pekuncen 3
    → TIDAK VALID |  | Nama tempat tidak terdeteksi; Nama/alamat kandidat tidak mengandung SPPG atau Satuan Pelayanan Peme

In [56]:
import pandas as pd

df = pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig")
print(f"Total baris : {len(df)}")
print("\nDistribusi Status:")
print(df["Status"].value_counts())

print("\nDistribusi GMaps_Source:")
print(df["GMaps_Source"].value_counts())

print("\nDistribusi GMaps_Is_SPPG_Like:")
print(df["GMaps_Is_SPPG_Like"].value_counts())

Total baris : 50

Distribusi Status:
Status
TIDAK VALID    49
VALID           1
Name: count, dtype: int64

Distribusi GMaps_Source:
GMaps_Source
search_direct_low_similarity      21
search_result_1                   16
search_result_3                    5
search_result_1_low_similarity     4
search_result_2                    3
search_result_3_low_similarity     1
Name: count, dtype: int64

Distribusi GMaps_Is_SPPG_Like:
GMaps_Is_SPPG_Like
YA       28
TIDAK    22
Name: count, dtype: int64


In [57]:
# Lihat baris TIDAK VALID beserta alasannya
invalid = df[df["Status"] == "TIDAK VALID"].copy()
print(f"Total TIDAK VALID: {len(invalid)}")
invalid[["Nama_SPPG", "GMaps_Nama", "GMaps_Name_Score", "GMaps_Distance_Meter", "GMaps_Error"]].head(20)

Total TIDAK VALID: 49


,Nama_SPPG,GMaps_Nama,GMaps_Name_Score,GMaps_Distance_Meter,GMaps_Error
0,SPPG Madiun Geger Sumberejo,SPPG MADIUN KEBONSARI REJOSARI,0.60,6533.8,Jarak koordinat 6533.8 m melebihi ambang 500.0 m
1,SPPG Malang Sumberpucung Karangkates,NaN,0.00,426.0,Nama tempat tidak terdeteksi; Nama/alamat kand...
2,SPPG Malang Ngantang Sumberagung 2,SPPG Sumberagung 01 Ngantang,0.50,913.2,Nama tidak cocok dengan CSV (score 0.50); Jara...
4,SPPG Gresik Driyorejo Driyorejo 2,SPPG Tanjungan,0.05,2048.1,Nama tidak cocok dengan CSV (score 0.05); Jara...
5,SPPG Kota Pasuruan Panggungrejo Pekuncen 3,NaN,0.00,1215.2,Nama tempat tidak terdeteksi; Nama/alamat kand...
6,SPPG Sampang Karangpenang Tlambah 4,Sppg karang penang tlambah 5,0.79,1788.9,Jarak koordinat 1788.9 m melebihi ambang 500.0 m
7,SPPG Sampang Ketapang Ketapang Timur 2,SPPG KETAPANG BARAT,0.43,4538.4,Nama tidak cocok dengan CSV (score 0.43); Jara...
8,SPPG Lamongan Lamongan Sukomulyo,NaN,0.00,899.2,Nama tempat tidak terdeteksi; Nama/alamat kand...
9,SPPG Blitar Nglegok Kedawung,SPPG BLITAR WLINGI WLINGI,0.56,12732.8,Jarak koordinat 12732.8 m melebihi ambang 500.0 m
10,SPPG Pacitan Pacitan Sirnoboyo,NaN,0.00,153.5,Nama tempat tidak terdeteksi; Nama/alamat kand...


## Debugging

In [58]:
# Debug satu baris secara manual (tanpa menjalankan pipeline penuh)
# Ubah index_debug ke nomor baris yang ingin diperiksa (0-based)

index_debug = 0  
test_row    = rows[index_debug]

print("=== Data CSV ===")
for k, v in test_row.items():
    print(f"  {k}: {v}")

print("\n=== Query yang akan dikirim ke GMaps ===")
print(build_search_query(test_row))

print("\n=== Name Score vs nama di output ===")
if index_debug < len(df):
    gmaps_name = df.iloc[index_debug]["GMaps_Nama"]
    score      = name_score(test_row.get("Nama_SPPG", ""), gmaps_name)
    print(f"  CSV  : {test_row.get('Nama_SPPG', '')}")
    print(f"  GMaps: {gmaps_name}")
    print(f"  Score: {score:.2f}")

=== Data CSV ===
  Provinsi: JAWA TIMUR
  Kab_Kota: MADIUN
  Kecamatan: GEGER
  Kelurahan_Desa: SUMBEREJO
  Alamat: Jl. Sultan Agung, Karanganyar, Sumberejo, Kec. Geger, Kabupaten Madiun, Jawa Timur 63171
  Nama_SPPG: SPPG Madiun Geger Sumberejo
  Longitude: 111.533
  Latitude: -7.72354

=== Query yang akan dikirim ke GMaps ===
SPPG Madiun Geger Sumberejo GEGER

=== Name Score vs nama di output ===
  CSV  : SPPG Madiun Geger Sumberejo
  GMaps: SPPG MADIUN KEBONSARI REJOSARI
  Score: 0.60
